In [2]:
# --- 0. Imports ---

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import folium
from folium.plugins import MarkerCluster

# Importation of the datasets

In [3]:
# --- 1. Loading historical data ---
path_hist_ancien = '/Users/quentingirard/Downloads/comptages-routiers-permanents.csv'
path_hist_new = '/Users/quentingirard/Downloads/comptages-routiers-permanents-2.csv'

if not os.path.exists(path_hist_ancien):
    raise FileNotFoundError(f"The old historical file is not found at: {path_hist_ancien}")
hist_df_old = pd.read_csv(path_hist_ancien, sep=";", encoding="utf-8")

if not os.path.exists(path_hist_new):
    raise FileNotFoundError(f"The historical file is not found at: {path_hist_new}")
hist_df_new = pd.read_csv(path_hist_new, sep=";", encoding="utf-8")

/var/folders/gl/j4h6g1555f7669vb8m7321jc0000gn/T/ipykernel_56145/4156254397.py:11: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  hist_df_new = pd.read_csv(path_hist_new, sep=";", encoding="utf-8")


In [4]:
# Loading API data

path_api = '/Users/quentingirard/Desktop/Ecole/3A_Ponts/MCNDU/Projet-comptage-routier/petit-entrepot-pour-MCNDU/data/traffic_2025-12_1.parquet'

if not os.path.exists(path_api):
    raise FileNotFoundError(f"The following file is not found: {path_api}")
api_df = pd.read_parquet(path_api, engine='pyarrow')

# Timerange Analysis

In [5]:
api_df["t_1h"] = pd.to_datetime(api_df["t_1h"], utc=True)
hist_df_old["Date et heure de comptage"] = pd.to_datetime(hist_df_old["Date et heure de comptage"], utc=True)
hist_df_new["Date et heure de comptage"] = pd.to_datetime(hist_df_new["Date et heure de comptage"], utc=True)

print(f"API Dataset : Date range: from {api_df['t_1h'].min()} to {api_df['t_1h'].max()}")
print(f"Old Historical Dataset :Date range: from {hist_df_old['Date et heure de comptage'].min()} to {hist_df_old['Date et heure de comptage'].max()}")
print(f"New Historical Dataset : Date range: from {hist_df_new['Date et heure de comptage'].min()} to {hist_df_new['Date et heure de comptage'].max()}")
# Supprime le recouvrement entre hist_df_old et hist_df_new

API Dataset : Date range: from 2025-12-08 16:00:00+00:00 to 2025-12-10 23:00:00+00:00
Old Historical Dataset :Date range: from 2024-10-01 03:00:00+00:00 to 2025-11-11 23:00:00+00:00
New Historical Dataset : Date range: from 2024-11-01 03:00:00+00:00 to 2025-12-16 23:00:00+00:00


# Deleting overlapping data

In [6]:
# --- 2. Suppression du recouvrement (overlapping) ---

# 1. Dans hist_df_old : on garde uniquement ce qui est strictement antérieur au début de hist_df_new
min_date_new = hist_df_new["Date et heure de comptage"].min()
hist_df_old = hist_df_old[hist_df_old["Date et heure de comptage"] < min_date_new]

# 2. Dans hist_df_new : on garde uniquement ce qui est strictement antérieur au début de api_df
min_date_api = api_df["t_1h"].min()
hist_df_new = hist_df_new[hist_df_new["Date et heure de comptage"] < min_date_api]

# --- 3. Vérification des nouveaux intervalles ---

print("Après nettoyage du recouvrement :")
print(f"Old Historical : Max date = {hist_df_old['Date et heure de comptage'].max()}")
print(f"New Historical : Min date = {hist_df_new['Date et heure de comptage'].min()} | Max date = {hist_df_new['Date et heure de comptage'].max()}")
print(f"API            : Min date = {api_df['t_1h'].min()}")

Après nettoyage du recouvrement :
Old Historical : Max date = 2024-11-01 02:00:00+00:00
New Historical : Min date = 2024-11-01 03:00:00+00:00 | Max date = 2025-12-08 15:00:00+00:00
API            : Min date = 2025-12-08 16:00:00+00:00


# Merge and export of Historical dataset

In [7]:
# --- 4. Fusion et tri des données historiques ---

# Concaténation des deux DataFrames
hist_df_combined = pd.concat([hist_df_old, hist_df_new], ignore_index=True)

# Tri par date décroissante (le plus récent en haut)
hist_df_combined = hist_df_combined.sort_values(by="Date et heure de comptage", ascending=False)

# Réinitialisation de l'index pour avoir une suite logique après le tri
hist_df_combined = hist_df_combined.reset_index(drop=True)

# --- 5. Vérification finale ---

print(f"Nombre de lignes total : {len(hist_df_combined)}")
print(f"Date la plus récente : {hist_df_combined['Date et heure de comptage'].max()}")
print(f"Date la plus ancienne : {hist_df_combined['Date et heure de comptage'].min()}")

# Aperçu des premières lignes
print("\nTop 5 des entrées les plus récentes :")
hist_df_combined[["Date et heure de comptage"]].head()

Nombre de lignes total : 28331770
Date la plus récente : 2025-12-08 15:00:00+00:00
Date la plus ancienne : 2024-10-01 03:00:00+00:00

Top 5 des entrées les plus récentes :


,Date et heure de comptage
0,2025-12-08 15:00:00+00:00
1,2025-12-08 15:00:00+00:00
2,2025-12-08 15:00:00+00:00
3,2025-12-08 15:00:00+00:00
4,2025-12-08 15:00:00+00:00


In [8]:
# --- 6. Exportation du dataset ---

# Définition du chemin de sortie complet
output_path = '/Users/quentingirard/Downloads/MCNDU_sorted_historical_dataset.csv'

# Exportation en CSV
hist_df_combined.to_csv(output_path, index=False, sep=';', encoding='utf-8')

print(f"✅ Dataset exporté avec succès ici : {output_path}")

✅ Dataset exporté avec succès ici : /Users/quentingirard/Downloads/MCNDU_sorted_historical_dataset.csv
